# Case Study: Solving Problems with Quantum Annealing (QA)
## From the Number Partitioning Problem to QUBO Formulation

**Estimated time: 20–30 minutes**  
**Format: Hands-on exercise in Google Colab, executed from top to bottom**

In this material, we use the simple and familiar **Number Partitioning Problem** to experience the following workflow.

1. Represent an optimization problem with an objective function  
2. Convert it to a QUBO using 0/1 variables  
3. Sample solutions of the QUBO  
4. Interpret the obtained solutions in the original problem

### Setup

In [ ]:
# Install the Ocean SDK SQA sampler (for Google Colab)
%pip install -q "dwave-samplers==1.8.0"

# Libraries used in this notebook
import numpy as np
import pandas as pd
from dwave.samplers import PathIntegralAnnealingSampler

SEED = 42
np.random.seed(SEED)

print("Setup complete")

# 1. Quantum Annealing and QUBO

## Quantum Annealing (QA)

In quantum annealing, candidate solutions are represented as **states** of a physical system.  
**The value of the objective function is treated as energy, and QA searches for low-energy states.**

Conceptually, the process is as follows.

1. Start with strong quantum fluctuations so that many states can be explored  
2. Gradually reduce the quantum fluctuations  
3. Finally, measure a state with a low objective-function value  

## QUBO

A common format that is easy to use as input to QA is **QUBO**. QUBO stands for  
**Quadratic Unconstrained Binary Optimization**.  
Each word describes one of the following properties.

- **Quadratic**: the objective function includes quadratic terms such as $x_ix_j$
- **Unconstrained**: equality and inequality constraints are not handled directly; necessary constraints are included in the objective function as penalty terms
- **Binary**: each variable takes either $0$ or $1$
- **Optimization**: we search for the binary-variable assignment that minimizes the objective function

The general form is shown below.  
$H(\mathbf{x})$ is called the Hamiltonian.

$$
H(\mathbf{x})
=
\sum_i Q_{ii}x_i
+
\sum_{i<j}Q_{ij}x_ix_j,
\qquad
x_i\in\{0,1\}
$$

- $x_i$: a variable that takes 0 or 1
- $Q_{ii}$: coefficient applied to a single variable
- $Q_{ij}$: coefficient applied to a pair of variables
- QA searches for a bit string that makes this Hamiltonian $H(\mathbf{x})$ small

Strictly speaking, the first term should also be written in quadratic form.  
However, because $x_i$ is a 0/1 variable, we can use $x_i^2=x_i$.

# 2. Number Partitioning Problem (NPP)

The Number Partitioning Problem splits several numbers into two groups so that the group sums are as close as possible.

Here, we consider a problem where **five camping items are divided between two cars so that the total weights of the cars are as close as possible**. Each item cannot be split and must be assigned to exactly one car.

| Item $i$ | Item type | Weight $a_i$ (kg) |
|---:|---|---:|
| 0 | Tent | 3 |
| 1 | Sleeping bag | 1 |
| 2 | Food | 4 |
| 3 | Cookware | 2 |
| 4 | Water | 2 |

Therefore, the weight vector is $[3,\ 1,\ 4,\ 2,\ 2]$. We assign a 0/1 variable $x_i$ to item $i$.

- $x_i=1$: item $i$ goes to Car A
- $x_i=0$: item $i$ goes to Car B

In [ ]:
# Weight information for each item
luggage = pd.DataFrame({
    "Item": ["Tent", "Sleeping bag", "Food", "Cookware", "Water"],
    "Weight (kg)": [3, 1, 4, 2, 2],
})
a = luggage["Weight (kg)"].to_numpy(dtype=int)

display(luggage)
print("Total weight:", int(a.sum()), "kg")
print("Ideal weight per car:", a.sum() / 2, "kg")

## Defining the objective function

Let the difference between the sums of Group A and Group B be

$$
D(\mathbf{x})
=
\sum_{i=1}^{N} a_i(2x_i-1)
$$

- When $x_i=1$, $2x_i-1=+1$: add the item to Group A
- When $x_i=0$, $2x_i-1=-1$: add the item to Group B

Therefore, $D(\mathbf{x})$ means

$$
D(\mathbf{x})
=
\text{sum of Group A}
-
\text{sum of Group B}
$$

We want to evaluate the size of this difference, so we square it.

$$
H(\mathbf{x})
=
\left(
\sum_{i=1}^{N} a_i(2x_i-1)
\right)^2
$$

- If the split is perfectly even, $H=0$
- If the difference is 2, $H=4$
- If the difference is 4, $H=16$

A QUBO is written using linear and quadratic terms of 0/1 variables.  
By squaring the difference, we remove the sign of the difference and obtain a quadratic expression that can be expanded into a QUBO.

In [ ]:
def partition_sums(x, values):
    """Return the two groups and their sums from a bit string x."""
    x = np.asarray(x, dtype=int)
    values = np.asarray(values, dtype=int)

    group_a = values[x == 1]
    group_b = values[x == 0]
    return group_a, group_b, int(group_a.sum()), int(group_b.sum())


def original_energy(x, values):
    """H(x) = (sum_i a_i(2x_i-1))^2"""
    x = np.asarray(x, dtype=int)
    values = np.asarray(values, dtype=int)
    difference = int(np.sum(values * (2 * x - 1)))
    return difference ** 2


# Example: x = [1, 0, 1, 0, 0]
x_example = np.array([1, 0, 1, 0, 0])
g_a, g_b, sum_a, sum_b = partition_sums(x_example, a)

print("x =", x_example.tolist())
print("Group A:", g_a.tolist(), "sum =", sum_a)
print("Group B:", g_b.tolist(), "sum =", sum_b)
print("H(x) =", original_energy(x_example, a))

# 3. Expanding into a QUBO

Let

$$
A=\sum_i a_i
$$

Then,

$$
\sum_i a_i(2x_i-1)
=
2\sum_i a_ix_i-A
$$

Therefore,

$$
H(\mathbf{x})
=
4\left(\sum_i a_ix_i\right)^2
-4A\sum_i a_ix_i
+A^2
$$

For 0/1 variables, $x_i^2=x_i$, so

$$
\left(\sum_i a_ix_i\right)^2
=
\sum_i a_i^2x_i
+
2\sum_{i<j}a_ia_jx_ix_j
$$

Thus,

$$
H(\mathbf{x})
=
\sum_i
\left(4a_i^2-4Aa_i\right)x_i
+
\sum_{i<j}
8a_ia_jx_ix_j
+
A^2
$$

The constant $A^2$ does not depend on $\mathbf{x}$, so it can be omitted when searching for the minimizing solution.

Therefore, the QUBO coefficients are

$$
Q_{ii}=4a_i^2-4Aa_i
$$

$$
Q_{ij}=8a_ia_j
\qquad(i<j)
$$

In [ ]:
def build_npp_qubo(values):
    """
    Build a dictionary Q representing
    E_QUBO(x) = sum_i Q[i,i] x_i + sum_{i<j} Q[i,j] x_i x_j.
    """
    values = np.asarray(values, dtype=int)
    total = int(values.sum())
    Q = {}

    # Linear terms (diagonal entries)
    for i, ai in enumerate(values):
        Q[(i, i)] = int(4 * ai**2 - 4 * total * ai)

    # Quadratic terms
    for i in range(len(values)):
        for j in range(i + 1, len(values)):
            Q[(i, j)] = int(8 * values[i] * values[j])

    return Q

Q = build_npp_qubo(a)

# QUBO matrix for display
Q_matrix = np.zeros((len(a), len(a)), dtype=int)
for (i, j), coeff in Q.items():
    Q_matrix[i, j] = coeff

display(pd.DataFrame(
    Q_matrix,
    index=[f"x{i}" for i in range(len(a))],
    columns=[f"x{i}" for i in range(len(a))]
))

# 4. Solving the QUBO with Ocean SDK SQA

This example has only five variables, so there are $2^5=32$ candidates and the solution can be found easily even by exhaustive search.  
However, in large-scale problems, the number of candidates grows as $2^N$, making exhaustive search rapidly difficult.

Instead, we often look for a “reasonably good solution that can be obtained in practical time,” even if it is not guaranteed to be optimal. This is called a heuristic approach, and QA is one such heuristic method.

Here, instead of using quantum annealing (QA) hardware, we use D-Wave Ocean SDK's `PathIntegralAnnealingSampler` to run **simulated quantum annealing (SQA)**.

SQA is a method that **approximately simulates the behavior of quantum annealing on a classical computer**. It searches for low-energy states by gradually reducing the transverse field (driver) that represents quantum fluctuations, while gradually strengthening the term that represents the problem QUBO (problem).

In this cell, we specify the following process to the Ocean SDK.

1. Start with a strong transverse field and a weak problem term  
2. Gradually reduce the transverse field and strengthen the problem term  
3. Read out the resulting low-energy bit strings  
4. Repeat independent runs multiple times to obtain a distribution of solutions

In [ ]:
def solve_npp_by_sqa(
    values,
    num_reads=100,
    num_schedule_points=100,
    seed=SEED,
):
    """Sample a number partitioning problem of any size using Ocean SDK SQA."""
    values = np.asarray(values, dtype=int)
    if values.ndim != 1 or len(values) == 0:
        raise ValueError("values must be a one-dimensional array with at least one number.")

    Q_local = build_npp_qubo(values)

    # Scale QUBO coefficients by a positive constant so that the minimizing solution does not change
    qubo_scale = max((abs(coeff) for coeff in Q_local.values()), default=1.0)
    if qubo_scale == 0:
        qubo_scale = 1.0
    Q_for_sqa = {key: coeff / qubo_scale for key, coeff in Q_local.items()}

    # Schedule: reduce the transverse field (driver) and strengthen the problem term
    problem_field = np.linspace(0.0, 5.0, num_schedule_points)
    driver_field = np.linspace(5.0, 0.0, num_schedule_points)

    sampler = PathIntegralAnnealingSampler()
    sampleset = sampler.sample_qubo(
        Q_for_sqa,
        num_reads=num_reads,
        num_sweeps=num_schedule_points,
        beta_schedule_type="custom",
        Hp_field=problem_field,
        Hd_field=driver_field,
        seed=seed,
    )

    records = []
    for sample in sampleset.samples():
        x_sample = np.array(
            [sample[i] for i in range(len(values))],
            dtype=int,
        )
        group_a, group_b, sum_a, sum_b = partition_sums(x_sample, values)
        records.append({
            "x": "".join(map(str, x_sample.tolist())),
            "H(x)": original_energy(x_sample, values),
            "Group A": group_a.tolist(),
            "Sum A": sum_a,
            "Group B": group_b.tolist(),
            "Sum B": sum_b,
        })

    result = pd.DataFrame(records)
    result = result.sort_values(["H(x)", "x"]).reset_index(drop=True)
    return result, sampleset, Q_local

In [ ]:
# Sample multiple times using Ocean SDK SQA
num_reads = 20
samples_df, sqa_sampleset, _ = solve_npp_by_sqa(
    a,
    num_reads=num_reads,
    seed=SEED,
)

print("Number of samples:", len(samples_df))
print("Minimum H(x) among samples:", int(samples_df["H(x)"].min()))
display(samples_df.head(8))

### Interpreting the best sample in the original problem
  
Finally, we convert the 0/1 values into items assigned to Car A and Car B.

In [ ]:
best_sample_string = samples_df.iloc[0]["x"]
best_sample = np.array([int(bit) for bit in best_sample_string])

group_a, group_b, sum_a, sum_b = partition_sums(best_sample, a)
car_a_items = luggage.loc[best_sample == 1, "Item"].tolist()
car_b_items = luggage.loc[best_sample == 0, "Item"].tolist()

print("Best bit string:", best_sample_string)
print("Car A:", car_a_items, "weights =", group_a.tolist(), "total =", sum_a, "kg")
print("Car B:", car_b_items, "weights =", group_b.tolist(), "total =", sum_b, "kg")
print("Total weight difference:", abs(sum_a - sum_b), "kg")
print("H(x):", original_energy(best_sample, a))

In this problem, if a bit string $\mathbf{x}$ is a solution, the bit string obtained by flipping all 0s and 1s represents the same split.  
Therefore, multiple optimal solutions may be found.  

QA does not return only one solution from a single run. By reading out multiple times, it can **sample a distribution of solutions**.

# 5. Summary

The workflow used here is common to many problems, not only the Number Partitioning Problem.

$$
\boxed{
\text{Real-world problem}
\rightarrow
\text{Formulate objectives and constraints}
\rightarrow
\text{Convert to QUBO}
\rightarrow
\text{Sample solutions}
\rightarrow
\text{Interpret in the real world}
}
$$

### Key points

- QA treats the objective function as energy and searches for low-energy states  
    By formulating a problem as a QUBO, we can solve optimization problems with QA
- QUBO represents a problem using $0/1$ variables and a quadratic form  
    For NPP, the objective function is the squared difference between the two group sums
- QA samples solutions multiple times  
    Sampling lets us observe several good solution patterns